# Fly Brain Lab: Quickstart

Voraussetzung: `flylab download` (Daten) und `pip install -e .`. Alles hier geht auch über die CLI (`flylab --help`).

In [ ]:
from flybrainlab.connectome import load_connectome, GroupResolver
from flybrainlab.connectome.loaders import sanity_check

c = load_connectome('malecns_v1.0')
sanity_check(c)['checks']

## Neuronengruppen

In [ ]:
g = GroupResolver(c)
[g.describe(n) for n in ['giant_fiber', 'lplc2_L', 'dna02_L', 'jo_ce']]

## Gehirn allein: LPLC2 aktivieren → Giant Fiber → TTMn (Sprungmuskel-Motoneuron)

In [ ]:
from flybrainlab.brain import LIFNetwork, LIFParams
net = LIFNetwork(c.W, LIFParams(w_syn=0.15, f_poi=458.333), seed=0)
net.set_poisson_rates(g.resolve('lplc2'), 100.0)
net.run(300)
r = net.rates_hz()
{k: r[g.resolve(k)].mean() for k in ['giant_fiber', 'ttmn']}

## Experiment mit Shuffle-Kontrolle

In [ ]:
from flybrainlab.experiments import run_experiment, compare_runs
res = run_experiment('looming', n_trials=4, label='intact', workers=4)
compare_runs(res['run_id'], res['control_run_id'], ['takeoff', 'gf_spikes'])

In [ ]:
from flybrainlab.viz.plots import plot_run
from IPython.display import Image
Image(str(plot_run(res['run_id'])))

## LLM-Experimentator

```bash
export ANTHROPIC_API_KEY=...
flylab agent "Hängt die Fluchtrichtung bei der Spinne an DNp11?" --budget 80
```
Ohne API-Key: `flylab study giant-fiber` (fester Plan, gleiche Tools und Bericht-Validierung).